# Mobile Money Adoption Trends Analysis

This notebook analyzes real-world mobile money and financial institution account adoption data from the **Our World in Data** API.

- **Data Source**: ourworldindata.org
- **Coverage**: 100+ countries from 2014-2022
- **Metrics**: Mobile money accounts, financial institution accounts, adoption shares

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mobile_money_project.data import load_mobile_money_data
from mobile_money_project.preprocessing import prepare_mobile_money_data
from mobile_money_project.analysis import summarize_mobile_money_trends
from mobile_money_project.modeling import train_mobile_money_forecast, forecast_mobile_money

# Load and prepare data from API
df = load_mobile_money_data('data/sample_mobile_money_data.csv')
df_prepared = prepare_mobile_money_data(df)

print(f"Dataset loaded: {len(df)} records across {df['country'].nunique()} countries")
print(f"Years: {df['year'].min()}-{df['year'].max()}")
df.head(10)

In [ ]:
# Trend analysis
summary = summarize_mobile_money_trends(df_prepared)
print("Global Trends Summary:")
for key, value in summary.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

In [ ]:
# Train forecast model
pipeline, backtest, y_test, metrics = train_mobile_money_forecast(df_prepared)
print("\nModel Performance:")
for metric, value in metrics.items():
    if metric not in ['features_used']:
        print(f"  {metric}: {value:.4f}" if isinstance(value, float) else f"  {metric}: {value}")

In [ ]:
# Generate future forecast and visualize
forecast = forecast_mobile_money(df_prepared, pipeline, forecast_horizon=12)
print("\nForecast (next 12 years):")
forecast_cols = [col for col in forecast.columns if col not in ['year', 'country']]
print(forecast[forecast_cols].tail(5))

# Visualization: Adoption trends by region
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# Plot 1: Mobile Money Share Trends
top_countries = df.groupby('country')['mobile_money_share'].mean().nlargest(5).index
for country in top_countries:
    country_data = df[df['country'] == country].sort_values('year')
    axes[0].plot(country_data['year'], country_data['mobile_money_share'], 
                label=country, marker='o', markersize=4)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Mobile Money Share (%)')
axes[0].set_title('Mobile Money Adoption Trends - Top 5 Countries')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot 2: Financial vs Mobile Money
df_global = df.groupby('year')[['mobile_money_share', 'financial_institution_share']].mean()
axes[1].plot(df_global.index, df_global['mobile_money_share'], label='Mobile Money Share', marker='o')
axes[1].plot(df_global.index, df_global['financial_institution_share'], label='Financial Institution Share', marker='s')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Adoption Share (%)')
axes[1].set_title('Global Average: Mobile Money vs Financial Institution Adoption')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Visualization complete!")